# Mapeamento de dados públicos da cana-de-açúcar

**Disciplina:** Visualização de Dados (PSAE00218) — UFPB · Departamento de Economia  
**Etapa 1:** Descoberta, persona e enquadramento do problema  
**Grupo:** Alex Tavares Cordeiro · Brenno Henrique Alves da Silva Costa · Gustavo Henrique Rocha Oliveira

Este notebook é a prova de que a base do projeto existe, é pública e tem a granularidade que o problema exige. Ele não produz visualização: coleta os dados na fonte, inspeciona a estrutura e documenta o que dá para calcular. É o material que sustenta os itens (h) e (i) do documento da Etapa 1.

A lógica aqui é a de *falhar cedo e barato*. Antes de desenhar qualquer tela, a gente confirma que o IBGE entrega cana em nível municipal e que a CONAB entrega a produção de açúcar e etanol por safra. Se algum desses dados não existisse na granularidade prometida, era melhor descobrir agora.

Para reproduzir, use o ambiente isolado do projeto:

```bash
uv run jupyter lab
```

In [1]:
import io
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

HEADERS = {"User-Agent": "Mozilla/5.0 (POC academica UFPB - Visualizacao de Dados)"}

## 1. IBGE — Produção Agrícola Municipal (Tabela 1612)

A PAM é a espinha dorsal do projeto. A Tabela 1612 traz área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção para cada lavoura temporária, e a cana-de-açúcar está entre elas (código 2696 no classificador de produtos). A consulta é feita pela API do SIDRA, que devolve JSON e não exige cadastro.

Começamos pelos metadados, para não trabalhar com código chutado.

In [2]:
meta = requests.get(
    "https://servicodados.ibge.gov.br/api/v3/agregados/1612/metadados", timeout=60
).json()

print(meta["nome"])
print("Periodicidade:", meta["periodicidade"])
print("\nVariáveis:")
for v in meta["variaveis"]:
    print(f"  [{v['id']}] {v['nome']} ({v.get('unidade')})")

print("\nProduto cana no classificador 81:")
for c in meta["classificacoes"]:
    if c["id"] == 81:
        for cat in c["categorias"]:
            if "cana-de" in cat["nome"].lower():
                print(f"  [{cat['id']}] {cat['nome']}")

Área plantada, área colhida, quantidade produzida, rendimento médio e valor da produção das lavouras temporárias
Periodicidade: {'frequencia': 'anual', 'inicio': 1974, 'fim': 2025}

Variáveis:
  [109] Área plantada (Hectares)
  [1000109] Área plantada - percentual do total geral (%)
  [216] Área colhida (Hectares)
  [1000216] Área colhida - percentual do total geral (%)
  [214] Quantidade produzida (Toneladas)
  [112] Rendimento médio da produção (Quilogramas por Hectare)
  [215] Valor da produção (Mil Cruzeiros [1974 a 1985, 1990 a 1992], Mil Cruzados [1986 a 1988], Mil Cruzados Novos [1989], Mil Cruzeiros Reais [1993], Mil Reais [1994 a 2025])
  [1000215] Valor da produção - percentual do total geral (%)

Produto cana no classificador 81:
  [2696] Cana-de-açúcar


Os metadados confirmam a série anual de 1974 a 2025 e os seis níveis territoriais (Brasil, região, UF, mesorregião, microrregião e município). Com isso, montamos uma função de coleta que traduz a resposta do SIDRA num dataframe limpo.

In [3]:
BASE_SIDRA = "https://apisidra.ibge.gov.br/values"
VARIAVEIS = "109,216,214,112,215"  # plantada, colhida, quantidade, rendimento, valor
CANA = "2696"


def coletar_sidra(nivel, localidades, periodos):
    url = (
        f"{BASE_SIDRA}/t/1612/n{nivel}/{localidades}"
        f"/v/{VARIAVEIS}/p/{periodos}/c81/{CANA}?formato=json"
    )
    dados = requests.get(url, timeout=120).json()
    df = pd.DataFrame(dados[1:])
    return df[["D1N", "D2N", "D3N", "V"]].rename(
        columns={"D1N": "local", "D2N": "variavel", "D3N": "periodo", "V": "valor"}
    )

### 1.1 Granularidade municipal (Paraíba)

O primeiro teste é o que mais importa para o produto: a cana aparece município a município na Paraíba? A notação `in n3 25` pede todos os municípios contidos na UF 25 (Paraíba).

In [4]:
municipios_pb = coletar_sidra("6", "in%20n3%2025", "2023")
print(f"Linhas retornadas: {len(municipios_pb)}")
municipios_pb.head(15)

Linhas retornadas: 1115


,local,variavel,periodo,valor
0,Água Branca - PB,Área plantada,2023,-
1,Água Branca - PB,Área colhida,2023,-
2,Água Branca - PB,Quantidade produzida,2023,-
3,Água Branca - PB,Rendimento médio da produção,2023,-
4,Água Branca - PB,Valor da produção,2023,-
5,Aguiar - PB,Área plantada,2023,5
6,Aguiar - PB,Área colhida,2023,5
7,Aguiar - PB,Quantidade produzida,2023,200
8,Aguiar - PB,Rendimento médio da produção,2023,40000
9,Aguiar - PB,Valor da produção,2023,32


Repare que alguns municípios trazem `-` no lugar do valor. Isso não é erro de coleta: o IBGE suprime ou não informa a produção onde ela é muito pequena. É uma limitação real da base e precisa entrar na documentação, porque afeta o cálculo de produtividade nesses municípios.

### 1.2 Comparação entre estados (benchmarking nacional)

O produto compara a Paraíba com outras regiões produtoras. Para isso, puxamos todas as UFs e ordenamos pela quantidade produzida.

In [5]:
ufs = coletar_sidra("3", "all", "2023")
ranking = ufs[ufs["variavel"] == "Quantidade produzida"].copy()
ranking["valor"] = pd.to_numeric(ranking["valor"], errors="coerce")
ranking = ranking.dropna(subset=["valor"]).sort_values("valor", ascending=False)
ranking[["local", "valor"]].head(12).reset_index(drop=True)

,local,valor
0,São Paulo,439098082
1,Minas Gerais,82544375
2,Goiás,81599588
3,Mato Grosso do Sul,51789876
4,Paraná,38586024
5,Mato Grosso,19060217
6,Alagoas,18982924
7,Pernambuco,16089574
8,Paraíba,7066802
9,Bahia,5651267


São Paulo concentra a maior parte da produção nacional, e a Paraíba aparece no bloco nordestino, ao lado de Alagoas e Pernambuco. Esse contraste é exatamente o que o produto quer tornar visível para a persona: onde a região dela se posiciona.

### 1.3 Evolução ao longo das safras (Paraíba)

Por fim, a série temporal do rendimento médio da Paraíba, que alimenta a leitura de evolução.

In [6]:
serie_pb = coletar_sidra("3", "25", "2003-2023")
rendimento = serie_pb[serie_pb["variavel"] == "Rendimento médio da produção"].copy()
rendimento["valor"] = pd.to_numeric(rendimento["valor"], errors="coerce")
rendimento[["periodo", "valor"]].tail(12).reset_index(drop=True)

,periodo,valor
0,2012,46556
1,2013,49927
2,2014,56404
3,2015,56446
4,2016,56290
5,2017,53620
6,2018,53554
7,2019,55552
8,2020,56478
9,2021,55654


## 2. CONAB — Série Histórica da Cana (camada industrial)

O IBGE cobre a lavoura, mas não a indústria. Quem produz açúcar e etanol e em que proporção é a CONAB que informa, por UF e por safra. É essa base que sustenta a leitura do *mix* açúcar/etanol, uma decisão estratégica que apareceu com força na entrevista com a persona.

O arquivo é um `.txt` separado por ponto e vírgula, codificado em latin-1.

In [7]:
url_conab = "https://portaldeinformacoes.conab.gov.br/downloads/arquivos/SerieHistoricaCana.txt"
txt = requests.get(url_conab, headers=HEADERS, timeout=120).content.decode("latin-1")
conab = pd.read_csv(io.StringIO(txt), sep=";", engine="python")
conab.columns = [c.strip() for c in conab.columns]
for c in conab.columns:
    if conab[c].dtype == object:
        conab[c] = conab[c].str.strip()

print(f"Linhas: {len(conab)}")
print("Colunas:", list(conab.columns))
conab.head()

Linhas: 470
Colunas: ['ano_agricola', 'dsc_safra_previsao', 'uf', 'produto', 'id_produto', 'area_plantada_mil_ha', 'producao_mil_t', 'dsc_situacao_levantamento', 'producao_acucar_mil_t', 'producao_etanol_anidro_mil_l', 'producao_etanol_hidratado_mil_l', 'producao_etanol_total_mil_l', 'produtcao_atr_kg_t']


,ano_agricola,dsc_safra_previsao,uf,produto,id_produto,area_plantada_mil_ha,producao_mil_t,dsc_situacao_levantamento,producao_acucar_mil_t,producao_etanol_anidro_mil_l,producao_etanol_hidratado_mil_l,producao_etanol_total_mil_l,produtcao_atr_kg_t
0,2005/06,UNICA,AL,CANA DE ACUCAR,4238,"402,1",23110.7,PREVISAO ...,2077.4,294667.9,278710.6,573378.5,137.2
1,2005/06,UNICA,AM,CANA DE ACUCAR,4238,"3,8",194.4,PREVISAO ...,22.7,486.3,367.2,853.5,130.2
2,2005/06,UNICA,BA,CANA DE ACUCAR,4238,55,3367.7,PREVISAO ...,242.3,229000.0,51675.3,280675.3,221.5
3,2005/06,UNICA,CE,CANA DE ACUCAR,4238,"35,1",1773.3,PREVISAO ...,0.0,569.3,27879.2,28448.5,27.2
4,2005/06,UNICA,ES,CANA DE ACUCAR,4238,"64,4",4243.4,PREVISAO ...,66.2,172000.9,79486.4,251487.3,119.6


Além de produção de cana, a base traz `producao_acucar_mil_t`, `producao_etanol_total_mil_l` e o **ATR** (açúcar total recuperável, em kg por tonelada), que é o indicador técnico usado para medir a qualidade da matéria-prima e decidir o destino da cana. Vamos isolar a Paraíba.

In [8]:
pb_conab = conab[conab["uf"] == "PB"][
    [
        "ano_agricola",
        "producao_mil_t",
        "producao_acucar_mil_t",
        "producao_etanol_total_mil_l",
        "produtcao_atr_kg_t",
    ]
]
pb_conab.tail(10).reset_index(drop=True)

,ano_agricola,producao_mil_t,producao_acucar_mil_t,producao_etanol_total_mil_l,produtcao_atr_kg_t
0,2017/18,5829.5,159.0,363898.0,136.5
1,2018/19,5589.1,117.5,382000.0,139.7
2,2019/20,6736.2,141.1,442746.0,135.4
3,2020/21,6242.1,143.8,406082.0,136.3
4,2021/22,6081.3,132.5,376834.0,130.6
5,2022/23,7302.4,169.2,452450.0,129.8
6,2023/24,7605.7,228.0,363057.0,124.4
7,2024/25,7486.6,308.0,332212.0,129.5
8,2025/26,6915.1,223.9,388485.0,128.2
9,2026/27,7290.3,142.0,456394.0,127.6


## 3. Indicadores que o produto vai calcular

As duas bases juntas permitem derivar os indicadores abaixo. Nenhum deles é lido pronto: todos vêm de cálculo sobre as variáveis brutas.

| Indicador | Fórmula | Unidade | Referência |
|-----------|---------|---------|------------|
| Produtividade | quantidade ÷ área colhida | t/ha | média estadual, nacional, safra anterior |
| Participação regional | produção da unidade ÷ produção do agregado × 100 | % | total estadual/nacional |
| Variação safra a safra | (ano t − ano t−1) ÷ ano t−1 × 100 | % | período anterior |
| Mix açúcar/etanol | razão entre litros de etanol e toneladas de açúcar (via ATR) | proporção | média nacional |
| Desvio vs. referência | (unidade − média do grupo) ÷ média do grupo × 100 | % | média da microrregião/UF |

Um exemplo de cálculo de produtividade recomposta a partir da PAM, para mostrar que o número fecha com o rendimento que o IBGE já publica.

In [9]:
base = coletar_sidra("3", "25", "2023")
base["valor"] = pd.to_numeric(base["valor"], errors="coerce")
pivot = base.pivot_table(index="periodo", columns="variavel", values="valor")

quantidade = pivot["Quantidade produzida"].iloc[0]
area_colhida = pivot["Área colhida"].iloc[0]
produtividade_calculada = quantidade / area_colhida  # t/ha
rendimento_ibge = pivot["Rendimento médio da produção"].iloc[0] / 1000  # kg/ha -> t/ha

print(f"Produtividade calculada:  {produtividade_calculada:.2f} t/ha")
print(f"Rendimento do IBGE:       {rendimento_ibge:.2f} t/ha")
print("Os dois valores batem, o que valida a fórmula do indicador.")

Produtividade calculada:  64.69 t/ha
Rendimento do IBGE:       64.69 t/ha
Os dois valores batem, o que valida a fórmula do indicador.


## 4. Limitações registradas

1. **Supressão em municípios pequenos (IBGE).** Onde a produção é baixa, o valor vem como `-`, o que impede o cálculo de produtividade nesses municípios.
2. **Calendários diferentes.** O IBGE trabalha com ano civil e a CONAB com safra (ano agrícola). Cruzar as duas exige uma regra explícita de alinhamento.
3. **Granularidades diferentes.** O IBGE chega ao município; a CONAB para na UF. A produção de açúcar e etanol não existe em nível municipal público.
4. **Defasagem.** A PAM fecha com cerca de um ano de atraso e a CONAB divulga estimativas por levantamento antes do número consolidado.

Essas quatro limitações vão para o item (h) do documento e para a seção de riscos de interpretação (item m), porque afetam diretamente como os números podem ser comparados.